# Bangla Self-Refine Pipeline V5.1 — Colab Runner
V5.1 adds numerical-qualifier auditing and a second-pass model-based source-support screen for System C. The second-pass verdict is a screening signal, not independent ground truth.
Run the local tests before making API calls. Start with one article, inspect the audit fields, then run the two-article smoke test.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload bangla_self_refine_pipeline_v5.zip
zip_name = next(iter(uploaded))
!unzip -q -o "$zip_name" -d /content
%cd /content/bangla_self_refine_pipeline


In [ ]:
!pip -q install -r requirements.txt


## Run local regression tests
These tests do not call the Gemini API. All should pass before the smoke test.

In [ ]:
!python -m unittest discover -s tests -v


## Add your API key safely
In Colab, open the Secrets panel, create `GEMINI_API_KEY`, and enable notebook access. Do not write the key directly into a notebook cell.

In [ ]:
from google.colab import userdata
import os
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ['GEMINI_MODEL'] = 'gemini-3.1-flash-lite'
assert os.environ['GEMINI_API_KEY'], 'Add GEMINI_API_KEY in Colab Secrets first.'


## Stage 1 — Download and sample XL-Sum
Pilot articles come from validation; main articles come from test.

In [ ]:
!python scripts/prepare_data.py --pilot-size 20 --main-size 150 --seed 42


## Stage 2 — One-article smoke test
Inspect V5.1 qualifier flags, exact quote validation, and semantic-support verdicts before increasing the sample.

In [ ]:
!python scripts/run_experiment.py --split pilot --limit 1 --max-rounds 3
!python scripts/inspect_v5_results.py --split pilot


## Optional — Two-article smoke test
After reviewing the first article, run the two-article smoke test in a clean results file or a fresh project copy.

In [ ]:
!python scripts/run_experiment.py --split pilot --limit 2 --max-rounds 3
!python scripts/inspect_v5_results.py --split pilot


In [ ]:
!python scripts/inspect_v51_results.py --split pilot
!python scripts/audit_reference_alignment.py --split pilot


## Stage 3 — Full pilot
Review the feedback and exact-quote checks, semantic-support verdicts, and qualifier audits. Freeze prompts before the main test run.

In [ ]:
!python scripts/run_experiment.py --split pilot
!python scripts/evaluate.py --split pilot


## Stage 4 — Main experiment
Run only after the pilot and prompt freeze. Re-running resumes from completed article IDs.

In [ ]:
!python scripts/run_experiment.py --split main
!python scripts/evaluate.py --split main


## Optional — BERTScore
This downloads a multilingual checkpoint and may be slow on CPU.

In [ ]:
!pip -q install -r requirements-bertscore.txt
!python scripts/evaluate.py --split main --bertscore --bertscore-model xlm-roberta-base


## Stage 5 — Create blinded human evaluation sheets
Do not share `annotations/admin_only/annotation_key.csv` with annotators.

In [ ]:
!python scripts/prepare_human_eval.py --n 60 --annotators 3 --seed 42
from google.colab import files
files.download('annotations/annotator_1.csv')
files.download('annotations/annotator_2.csv')
files.download('annotations/annotator_3.csv')


After annotators complete their CSVs, upload the completed files back into `annotations/` with the same filenames, then run `!python scripts/analyze_human_eval.py`.